In [2]:
# pip freeze > requirements.txt (to save the current environment)

# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from datetime import datetime
# TODO: Model import goes here


# NBA API Endpoints
from nba_api.stats.endpoints import playercareerstats
from nba_api.stats.endpoints import commonplayerinfo
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.endpoints import boxscoresummaryv2
from nba_api.stats.endpoints import teamgamelog




## Workflow:
1. [⚠️FRONTEND] Get player name 
2. ✅ Get player ID from player name
3. Fetch career stats, gamelog, team gamelog using player ID
    Stats to fetch:
    - ✅ Career stats (averaged stats) [Need for predicting over/under prop]
    - Game log (past season games; grab prev season if <25% season completed; grab past 2 seasons games against certain team) [Need for dataset]
        - Core stats (PTS, REBOUNDS, ASSISTS, STEALS, 3PT MADE, DOUBLE-DOUBLES, TRIPLE-DOUBLES) [For dataset]
    - Box score summary (injury report) [Need for injury report]
    
4. Form dataset from all fetched endpoints
5. Preprocess the dataset
6. Train-test split (or any other splitting method)
7. Train the model + evaluate (fine tune)
8. Run predictions
9. [⚠️LLM] Get LLM to summarize the predictions


In [3]:
# List of all players in the NBA currently (will be used to autofill in search bar)
from nba_api.stats.static.players import get_players
players = pd.DataFrame(get_players())

# Function to get player ID from player name
def get_player_id(player_name):
    active_players = players[players['is_active']== True]

    # Retrieve player ID provided player name
    player_id = active_players[active_players['full_name'] == player_name]['id'].values[0]
    return player_id

test_name = 'Cade Cunningham' # TODO: Get user input from frontend (MUI autocomplete)

player_id = get_player_id(test_name) # Get player ID from player name
print(test_name, player_id) # Print player ID
print("Player Name Retrieved")


Cade Cunningham 1630595
Player Name Retrieved


In [4]:
# Fetching avg career data
avg_career_stats = playercareerstats.PlayerCareerStats(player_id=player_id, per_mode36="PerGame").get_data_frames()[3] # Averaged career stats for regular season [Need for predicting prop]
print("Average career stats retrieved")


Average career stats retrieved


In [5]:
# Function to get current and previous NBA season based on the current date
def get_season():
    today = datetime.today()
    year = today.year
    month = today.month

    if month >= 10:  # October to December: new season starts
        start_year = year
    else:  # January to September: still part of the previous season
        start_year = year - 1

    current_season = f"{start_year}-{str(start_year + 1)[-2:]}"
    previous_season = f"{start_year - 1}-{str(start_year)[-2:]}"
    
    return current_season, previous_season

current_season, previous_season = get_season()

commoninfo = commonplayerinfo.CommonPlayerInfo(player_id=player_id) # Player info
team_id = commoninfo.get_data_frames()[0]['TEAM_ID'].values[0]

# Function to get team game logs
def get_team_games(team_id):
    curr_team_gamelog = teamgamelog.TeamGameLog(team_id=team_id, season=current_season, season_type_all_star="Regular Season").get_data_frames()[0] # Team game log
    
    # Check if team has played at least 21 games in the current season (1/4 season)
    if curr_team_gamelog.count()['Game_ID'] <= 20:
        print("Not enough games played in current season, using previous season data as well")
        prev_team_gamelog = teamgamelog.TeamGameLog(team_id=team_id, season=previous_season, season_type_all_star="Regular Season").get_data_frames()[0]
        team_gamelog = pd.concat([curr_team_gamelog, prev_team_gamelog], ignore_index=True) # Combine current and previous season game logs
    
    else:
        print("Enough games played in current season, using only current season data")
        team_gamelog = curr_team_gamelog
    
    return team_gamelog[['Game_ID', "MATCHUP", "WL"]] # DataFrame of non-player stats for each game


team_games = get_team_games(team_id) # Get team games

# Function to get player game logs
def get_player_gamelog(player_id, team_id):
    team_gamelog = teamgamelog.TeamGameLog(team_id=team_id, season=current_season, season_type_all_star="Regular Season").get_data_frames()[0] # Team game log
    
    if team_gamelog.count()['Game_ID'] <= 20:
        prev_gamelog = playergamelog.PlayerGameLog(player_id=player_id, season=previous_season, season_type_all_star="Regular Season").get_data_frames()[0] # Previous season game log
        curr_gamelog = playergamelog.PlayerGameLog(player_id=player_id, season=current_season, season_type_all_star="Regular Season").get_data_frames()[0] # Current season game log
        gamelog = pd.concat([curr_gamelog, prev_gamelog], ignore_index=True) # Combine current and previous season game logs

    else:
        gamelog = playergamelog.PlayerGameLog(player_id=player_id, season=current_season, season_type_all_star="Regular Season").get_data_frames()[0] # Game log

    return gamelog

player_gamelog = get_player_gamelog(player_id=player_id, team_id=team_id) # Get player game log

gamelog = pd.merge(player_gamelog, team_games, how='right', on=['Game_ID','MATCHUP', 'WL']) # Merge player gamelog with team game log
gamelog.set_index('Game_ID', inplace=True) # Set Game_ID as index
print("Player game log retrieved")


Enough games played in current season, using only current season data
Player game log retrieved


In [ ]:
# Function to get box score summary for inactive players
# TODO: Write a function to retrieve it for all the games in the gamelog 
# and show whether or not the player played as a new column

import time

def player_inactive(player_id):
    inactive_games = [] # List to store inactive games

    # Iterate through each box score summary in the gamelog to find games where the player was inactive
    for gameId in team_games['Game_ID']:
        boxscore = boxscoresummaryv2.BoxScoreSummaryV2(game_id=str(gameId)).get_data_frames()[3] # Inactive players
        time.sleep(1)

        if player_id in boxscore['PLAYER_ID'].values:
            print(f"Inactive for {gameId}")
            inactive_games.append(gameId)

    return inactive_games # Return list of inactive games

player_inactive(player_id) # Check if player is inactive for any game in the gamelog

# TODO: Populate the main dataset with this info and the remaining rows count as Did-Not-Dress days

Inactive for 0022401121
Inactive for 0022401108
Inactive for 0022401084
Inactive for 0022401067
Inactive for 0022400293
Inactive for 0022400278
Inactive for 0022400265


In [ ]:
# Function to grab additional rivalry matches for the player vs opponent team
# TODO: Write this function